# Init

In [ ]:
pip install mplcursors

# Wybielanie

In [ ]:
import cv2
import numpy as np

def whiten_scan(image_path, output_path):
    # 1. Wczytanie obrazu
    img = cv2.imread(image_path)

    # 2. Konwersja do skali szarości
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 3. Usuwanie szumu (lekki blur pomaga AI nie widzieć "ziaren" papieru)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # 4. Progowanie adaptacyjne
    # cv2.ADAPTIVE_THRESH_GAUSSIAN_C oblicza próg dla każdego piksela osobno
    # 115 to rozmiar bloku (musi być nieparzysty), 15 to stała odejmowana od średniej
    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 115, 33
    )

    # 5. Opcjonalne: czyszczenie drobnych "paproszków" (morfologia)
    kernel = np.ones((1, 1), np.uint8)
    clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # 6. Zapisanie efektu
    cv2.imwrite(output_path, clean)
    print(f"Gotowe! Wybielony skan zapisano jako: {output_path}")



In [ ]:
import cv2
import numpy as np
import os # Dodano import modułu os do sprawdzania istnienia plików
path = '19'
whiten_scan(image_path='final_ocr_ready.jpg', output_path=f'19xxl.jpg')

# Dekompozycja na kolory

In [ ]:
import cv2
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def process_document_classic_pca_jpg(image_path):
    # 1. Wczytanie obrazu
    img = cv2.imread(image_path)
    if img is None:
        return print("Błąd: Nie znaleziono pliku.")

    # 2. Rozszerzenie cech (RGB + HSV + LAB)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    combined_data = np.concatenate((img, hsv, lab), axis=2)
    h, w, c = combined_data.shape
    pixels = combined_data.reshape(-1, c)

    # Skalowanie danych
    scaler = StandardScaler()
    pixels_scaled = scaler.fit_transform(pixels)

    # 3. PCA - ustawiamy na 5 komponentów dla lepszej separacji
    n_comps = 5
    pca = PCA(n_components=n_comps)
    pca_result = pca.fit_transform(pixels_scaled)

    layers = []
    for i in range(n_comps):
        comp = pca_result[:, i]
        comp_norm = (255 * (comp - comp.min()) / (comp.max() - comp.min())).astype(np.uint8)
        layer_img = comp_norm.reshape(h, w)

        # Auto-inwersja: jeśli tło jest ciemne, odwracamy (zakładamy białą kartkę)
        if np.mean(layer_img[0:30, 0:30]) < 127:
            layer_img = cv2.bitwise_not(layer_img)
        layers.append(layer_img)

    # 4. Klasyczna identyfikacja (Punktacja)
    scores = []
    for idx, layer in enumerate(layers):
        # Czerwona kreska (korelacja z kanałem R)
        red_ref = cv2.subtract(cv2.subtract(img[:,:,2], img[:,:,1]), img[:,:,0])
        red_score = np.corrcoef(layer.flatten(), red_ref.flatten())[0, 1]

        # Kratka (Hough)
        edges = cv2.Canny(layer, 50, 150)
        lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=15)
        grid_score = len(lines) if lines is not None else 0

        # Tekst (Liczba konturów)
        _, thresh = cv2.threshold(layer, 160, 255, cv2.THRESH_BINARY_INV)
        cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        text_score = len([c for c in cnts if 10 < cv2.contourArea(c) < 1000])

        scores.append({'idx': idx, 'red': abs(red_score), 'grid': grid_score, 'text': text_score, 'img': layer})

    # 5. Wybór najlepszych i zapis wszystkich do JPG
    best_text = max(scores, key=lambda x: x['text'])
    best_grid = max(scores, key=lambda x: x['grid'])
    best_red = max(scores, key=lambda x: x['red'])

    # Parametry zapisu JPG
    jpg_params = [int(cv2.IMWRITE_JPEG_QUALITY), 95]

    cv2.imwrite("WYNIK_TEKST.jpg", best_text['img'], jpg_params)
    cv2.imwrite("WYNIK_KRATKA.jpg", best_grid['img'], jpg_params)
    cv2.imwrite("WYNIK_CZERWONY.jpg", best_red['img'], jpg_params)

    # Zapisujemy też wszystkie warstwy raw jako backup
    for s in scores:
        cv2.imwrite(f"warstwa_raw_{s['idx']}.jpg", s['img'], jpg_params)

    print("-" * 30)
    print("ZAKOŃCZONO: Wszystkie warstwy zapisane jako .jpg")
    print(f"Najlepszy tekst: warstwa {best_text['idx']}")
    print(f"Najlepsza kratka: warstwa {best_grid['idx']}")
    print("-" * 30)

process_document_classic_pca_jpg('5.jpg')

# Wymuszanie liczb

In [ ]:

from transformers import LogitsProcessor, LogitsProcessorList
class OnlyDigitsLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer):
        self.allowed_token_ids = set(tokenizer.all_special_ids)

        # Przeszukujemy słownik modelu
        for token, token_id in tokenizer.get_vocab().items():
            # TrOCR (RoBERTa) używa znaku 'Ġ' dla spacji, usuwamy go do testu
            clean_token = token.replace('Ġ', '')
            # Jeśli token składa się TYLKO z cyfr, pozwalamy na niego
            if clean_token.isdigit():
                self.allowed_token_ids.add(token_id)

        self.allowed_token_ids = torch.tensor(list(self.allowed_token_ids))

    def __call__(self, input_ids, scores):
        # Tworzymy maskę, gdzie wszystko ma szansę -nieskończoność (0%)
        mask = torch.full_like(scores, -float('inf'))
        # Odblokowujemy tylko nasze cyfrowe tokeny
        allowed_ids = self.allowed_token_ids.to(scores.device)
        mask[:, allowed_ids] = 0

        return scores + mask

# Tworzymy instancję i listę procesorów dla modelu
digits_processor = OnlyDigitsLogitsProcessor(processor.tokenizer)
logits_processor_list = LogitsProcessorList([digits_processor])

# Zwykłe odczytywanie

In [ ]:
import easyocr
import cv2
from transformers import LogitsProcessor, LogitsProcessorList
import torch
import re
import numpy as np
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from google.colab.patches import cv2_imshow

# 1. Inicjalizacja modeli z optymalizacją sprzętową
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"INFO: Używam: {device.upper()}")

# Inicjalizacja EasyOCR
reader_detector = easyocr.Reader(['pl'], gpu=True)

# Inicjalizacja TrOCR w trybie FP16 (Half Precision) dla 2x szybszej inferencji na GPU
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten").to(device)

if device == "cuda":
    model.half() # Przejście na FP16 - kluczowe dla szybkości na T4/A100

def process_optimized_ocr(image_path):
    img_cv = cv2.imread(image_path)
    if img_cv is None: return []
    h, w = img_cv.shape[:2]

    # --- KROK 1: ROI CROPPING (Oszczędzamy 50% czasu detekcji) ---
    q = h // 5
    top_roi = img_cv[0:q, :]
    bottom_roi = img_cv[3*q:h, :]

    # Łączymy górę i dół w jeden mniejszy obraz, by wywołać EasyOCR tylko RAZ
    combined_roi = np.vstack((top_roi, bottom_roi))

    # Detekcja tylko na wybranym obszarze
    #results_detector = reader_detector.readtext(combined_roi)
    results_detector = reader_detector.readtext(top_roi)


    crops_for_trocr = []
    found_numbers = []
    bboxes = []

    # --- KROK 2: PRZYGOTOWANIE BATCHA ---
    margin = 2
    for (bbox, text_detected, prob) in results_detector:
        tl, br = bbox[0], bbox[2]
        x1, y1 = max(0, int(tl[0])-margin), max(0, int(tl[1])-margin)
        x2, y2 = min(combined_roi.shape[1], int(br[0])+margin), min(combined_roi.shape[0], int(br[1])+margin)

        crop = combined_roi[y1:y2, x1:x2]
        if crop.size == 0: continue

        # Konwersja na format PIL i dodanie do listy batcha
        img_crop_pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        crops_for_trocr.append(img_crop_pil)
        bboxes.append((x1, y1, x2, y2))

    # --- KROK 3: BATCH INFERENCE (Największy zysk prędkości) ---
    if crops_for_trocr:
        # Przetwarzamy wszystkie wycinki jednocześnie
        pixel_values = processor(images=crops_for_trocr, return_tensors="pt").pixel_values.to(device)

        if device == "cuda":
            pixel_values = pixel_values.half() # Dopasowanie do modelu FP16

        # Generowanie tekstu dla całego batcha naraz
        with torch.no_grad():
            generated_ids = model.generate(
                pixel_values,
                logits_processor=logits_processor_list
            )

            decoded_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Analiza wyników
        for i, text in enumerate(decoded_texts):
            clean_text = text.replace(" ", "")
            numbers = re.findall(r'\d+', clean_text)
            if numbers:
                found_numbers.append("".join(numbers))

            # Rysowanie na obrazie poglądowym (combined_roi)
            x1, y1, x2, y2 = bboxes[i]
            cv2.rectangle(combined_roi, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(combined_roi, clean_text, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # Wizualizacja
    print(f"Znalezione liczby: {found_numbers}")
    cv2_imshow(cv2.resize(combined_roi, (int(w*0.5), int(h*0.25*2*0.5)))) # Skalowanie podglądu

    return found_numbers


# Konsument-producent

In [ ]:
import easyocr
import cv2
import torch
import re
import numpy as np
import threading
import queue
import time
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from google.colab.patches import cv2_imshow
from transformers import LogitsProcessor, LogitsProcessorList


# 1. Inicjalizacja modeli z optymalizacją sprzętową
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"INFO: Używam: {device.upper()}")

# Inicjalizacja EasyOCR
reader_detector = easyocr.Reader(['pl'], gpu=True)

# Inicjalizacja TrOCR w trybie FP16 (Half Precision) dla 2x szybszej inferencji na GPU
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten").to(device)

if device == "cuda":
    model.half() # Przejście na FP16 - kluczowe dla szybkości na T4/A100

# --- KOLEJKA PRODUCENT-KONSUMENT ---
task_queue = queue.Queue(maxsize=5)
final_results = {}

# --- PRODUCENT (Wczytuje, tnie, robi EasyOCR) ---
def producer(image_paths):
    for image_path in image_paths:
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            print(f"Brak pliku: {image_path}")
            continue

        h, w = img_cv.shape[:2]

        # --- KROK 1: ROI CROPPING (Twoja oryginalna logika) ---
        q = h // 5
        top_roi = img_cv[0:q, :]
        bottom_roi = img_cv[3*q:h, :]

        combined_roi = np.vstack((top_roi, bottom_roi))

        # Detekcja tylko na wybranym obszarze (Twoja logika)
        results_detector = reader_detector.readtext(top_roi)

        crops_for_trocr = []
        bboxes = []

        # --- KROK 2: PRZYGOTOWANIE BATCHA ---
        margin = 2
        for (bbox, text_detected, prob) in results_detector:
            tl, br = bbox[0], bbox[2]
            x1, y1 = max(0, int(tl[0])-margin), max(0, int(tl[1])-margin)
            x2, y2 = min(combined_roi.shape[1], int(br[0])+margin), min(combined_roi.shape[0], int(br[1])+margin)

            crop = combined_roi[y1:y2, x1:x2]
            if crop.size == 0: continue

            img_crop_pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            crops_for_trocr.append(img_crop_pil)
            bboxes.append((x1, y1, x2, y2))

        # Jeśli cokolwiek znaleziono, przygotuj dla modelu i wrzuć do kolejki
        if crops_for_trocr:
            # Wstępne procesowanie jeszcze na CPU
            pixel_values = processor(images=crops_for_trocr, return_tensors="pt").pixel_values
            task_queue.put((image_path, pixel_values, combined_roi, bboxes, w, h))
        else:
            final_results[image_path] = []
            print(f"[{image_path}] Brak tekstu do przetworzenia.")

    # Sygnał końca pracy dla konsumenta
    task_queue.put(None)

# --- KONSUMENT (Odpala TrOCR na GPU i rysuje) ---
def consumer():
    while True:
        item = task_queue.get()
        if item is None: # Otrzymano sygnał końca
            break

        image_path, pixel_values, combined_roi, bboxes, w, h = item
        found_numbers = []

        # --- KROK 3: BATCH INFERENCE ---
        pixel_values = pixel_values.to(device)

        if device == "cuda":
            pixel_values = pixel_values.half() # Dopasowanie do modelu FP16

        with torch.no_grad():
            generated_ids = model.generate(
                pixel_values,
                logits_processor=logits_processor_list
            )
            decoded_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Analiza wyników
        for i, text in enumerate(decoded_texts):
            clean_text = text.replace(" ", "")
            numbers = re.findall(r'\d+', clean_text)
            if numbers:
                found_numbers.append("".join(numbers))


        final_results[image_path] = found_numbers
        task_queue.task_done()
        print("Zrobiono")

# --- URUCHOMIENIE (Przykład dla listy plików) ---
lista_plikow = [f'{i}d.jpg' for i in range (1, 44)]

start_time = time.time()

# Uruchomienie wątków
thread_prod = threading.Thread(target=producer, args=(lista_plikow,))
thread_cons = threading.Thread(target=consumer)

thread_prod.start()
thread_cons.start()

thread_prod.join()
thread_cons.join()


print(f"\n⏱ Czas całkowity: {time.time() - start_time:.2f}s")
print(final_results)

## Uruchomienie odczytywania

In [ ]:
import time

start = time.time()
result = process_optimized_ocr("19xxl.jpg")
results2 = [res for res in result if int(res) >= 100 and int(res) < 10000000 ]

print(f"Czas wykonania: {time.time() - start:.2f}s")
print(results2)

# Nowe podejście

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files


def process_for_ocr(image_path):
    # Wczytanie obrazu
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. Progowanie (zakładamy wybieloną kartkę - szukamy ciemnych elementów)
    # Używamy progowania binarnego odwróconego
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

    # 3. Wykrywanie linii metodą Hougha (odporna na skosy)
    line_mask = np.zeros_like(gray)

    lines = cv2.HoughLinesP(thresh, 1, np.pi/180, threshold=50,
                            minLineLength=70, maxLineGap=5)

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Rysujemy linie na masce (nieco grubiej - grubosć 3)
            cv2.line(line_mask, (x1, y1), (x2, y2), 255, 3)

    result = cv2.inpaint(img, line_mask, 3, cv2.INPAINT_TELEA)
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)


    # Zapis pliku wynikowego
    cv2.imwrite('zrdk.jpg', cv2.cvtColor(result_rgb, cv2.COLOR_RGB2BGR))
    print("Gotowe")

process_for_ocr('zr.jpg')

# Gemini

In [ ]:
pip install openai

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`.

In [ ]:
# Used to securely store your API key
from google.colab import userdata

# Fetch the API key from Colab Secrets
API_KEY = userdata.get('GOOGLE_API_KEY')

# Odczytywanie

In [ ]:
import os
import json
import time
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor
from IPython.display import clear_output  # Przydatne w Colabie do czyszczenia ekranu
 # Pamiętaj o wklejeniu swojego klucza!

# ============================================================
# COLAB - GEMINI + OCR ZDJĘĆ
# ============================================================

!pip install -q -U google-genai pillow

import os
import json
import time
import random

from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
from google import genai
from google.genai import types


# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')


# ============================================================
# 2. GEMINI API
# ============================================================

# W Colab:
# ikona kluczyka / Secrets
# dodaj:
#
# GEMINI_API_KEY = TWÓJ_KLUCZ
#
# Jeśli nie chcesz używać Secrets, możesz wpisać klucz
# bezpośrednio poniżej.


API_KEY = os.environ.get("GEMINI_API_KEY")  # klucz z Colab Secrets / zmiennej środowiskowej
# Jeśli używasz klucza bezpośrednio:
# API_KEY = os.environ.get("GEMINI_API_KEY")  # klucz z Colab Secrets / zmiennej środowiskowej

if not API_KEY:
    raise RuntimeError(
        "Brak GEMINI_API_KEY.\n"
        "W Colab wejdź w Secrets i dodaj GEMINI_API_KEY."
    )

client = genai.Client(api_key=API_KEY)

# Aktualny model Gemini
MODEL_ID = "gemini-3.6-flash"

print("Gemini client utworzony.")


# ============================================================
# 3. TEST POŁĄCZENIA Z GEMINI
# ============================================================

try:
    test_response = client.models.generate_content(
        model=MODEL_ID,
        contents="Odpowiedz tylko: OK"
    )

    print("Odpowiedź Gemini:", test_response.text)

except Exception as e:
    print("\n!!! GEMINI NIE DZIAŁA !!!")
    print(type(e).__name__)
    print(str(e))
    raise


# ============================================================
# 4. USTAWIENIA
# ============================================================

PUNKTY = "punkty"

FOLDER_SCIEZKA = "/content/drive/MyDrive/zdj-egz-2/"

# 40 jest za dużo przy limitach API.
# Sama logika programu pozostaje taka sama,
# tylko ograniczamy liczbę jednoczesnych połączeń.
MAX_WORKERS = 5


# ============================================================
# 5. PROMPTY
# ============================================================

PROMPT_imienazwisko = '''
Wyciągnij dane: punkty (0-30).

Zwróć czysty obiekt JSON o polach:

{
    "punkty": null,
    "imie_i_nazwisko": null,
    "nr_indeksu": null,
    "tytul": false
}

- "imie_i_nazwisko" - imię i nazwisko znalezione na stronie.
  Jeżeli nie znajdziesz, zwróć null.

- "nr_indeksu" - liczba sześciocyfrowa.
  Jeżeli nie znajdziesz, zwróć null.

- "tytul" - True jeżeli znalazłeś imię i nazwisko LUB nr indeksu.
  W przeciwnym przypadku False.

Na tym etapie pole "punkty" zawsze ustaw na null.

Uważaj przy rozpoznawaniu tekstu.
'''


PROMPT_punkty = '''
Wyciągnij dane: punkty (0-30).

Jest to liczba lub cyfra zapisana osobno od rozwiązania.

Może być wzięta w kółko.
Uważaj - liczba może nachodzić na kółko.

Jeżeli nie jesteś pewny, a liczba jest wzięta w kółku,
spróbuj zgadnąć która część jest liczbą, a która kółkiem,
i rozpoznaj tylko tę część, która wygląda jak liczba.

Uważaj zwłaszcza na 0 -
w kółku może wyglądać jak kółko w kółku.

Jeżeli uważasz, że na danej stronie nie ma zapisanej liczby punktów,
zwróć null.

Zwróć WYŁĄCZNIE JSON:

{
    "punkty": liczba_lub_null
}
'''


# ============================================================
# 6. FUNKCJA GEMINI Z RETRY
# ============================================================

def gemini_request(contents, max_retries=7):

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json"
                )
            )

            return response

        except Exception as e:

            error = str(e)

            # Błędy, przy których warto ponowić request
            retryable = (
                "429" in error
                or "RESOURCE_EXHAUSTED" in error
                or "503" in error
                or "UNAVAILABLE" in error
                or "500" in error
                or "INTERNAL" in error
            )

            if not retryable:
                raise

            if attempt == max_retries - 1:
                raise

            # exponential backoff
            wait = min(
                60,
                2 ** attempt + random.uniform(0, 1)
            )

            print(
                f"Gemini limit/błąd chwilowy. "
                f"Próba {attempt + 1}/{max_retries}. "
                f"Czekam {wait:.1f}s..."
            )

            time.sleep(wait)


# ============================================================
# 7. PARSOWANIE JSON
# ============================================================

def parse_json(response):

    text = response.text.strip()

    try:
        data = json.loads(text)

    except json.JSONDecodeError:

        # awaryjnie wycinamy JSON
        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1:
            raise ValueError(
                f"Gemini zwrócił niepoprawny JSON:\n{text}"
            )

        data = json.loads(
            text[start:end + 1]
        )

    if isinstance(data, list):
        data = data[0]

    return data


# ============================================================
# 8. PRZETWARZANIE JEDNEGO PLIKU
# ============================================================

def przetwarzaj_plik(i):

    photo_path = f"{FOLDER_SCIEZKA}{i}.jpg"

    if not os.path.exists(photo_path):

        print(
            f"Próbowano wczytać nieistniejący plik: "
            f"{photo_path}"
        )

        return None

    try:

        img = Image.open(photo_path).convert("RGB")

        # ----------------------------------------------------
        # REQUEST 1 - imię / nazwisko / indeks
        # ----------------------------------------------------

        response = gemini_request(
            [
                PROMPT_imienazwisko,
                img
            ]
        )

        dane = parse_json(response)

        if isinstance(dane, list):
            dane = dane[0]

        # ----------------------------------------------------
        # tytul
        # ----------------------------------------------------

        imie = dane.get("imie_i_nazwisko")
        indeks = dane.get("nr_indeksu")

        dane["tytul"] = (
            imie is not None
            or indeks is not None
        )

        # ----------------------------------------------------
        # REQUEST 2 - punkty
        # ----------------------------------------------------

        if dane["tytul"]:

            response_punkty = gemini_request(
                [
                    PROMPT_punkty,
                    img
                ]
            )

            try:

                wartosc = parse_json(
                    response_punkty
                ).get(PUNKTY)

                if (
                    wartosc is None
                    or str(wartosc).lower() == "none"
                ):

                    dane[PUNKTY] = None

                else:

                    dane[PUNKTY] = float(wartosc)

                    # dodatkowa kontrola
                    if not 0 <= dane[PUNKTY] <= 30:
                        dane[PUNKTY] = None

            except Exception:

                dane[PUNKTY] = None

        else:

            dane[PUNKTY] = None

        # ----------------------------------------------------
        # PHOTO
        # ----------------------------------------------------

        dane["photo"] = photo_path

        print(
            f"Plik {i}.jpg przetworzony | "
            f"{dane.get('imie_i_nazwisko')} | "
            f"{dane.get('nr_indeksu')} | "
            f"punkty={dane.get('punkty')}"
        )

        return dane

    except Exception as e:

        print(
            f"Błąd w pliku {i}.jpg: "
            f"{type(e).__name__}: {e}"
        )

        return None


# ============================================================
# 9. PRZETWARZANIE WIELU PLIKÓW
# ============================================================

def przetworz_wszystkie(numery_plikow):

    wyniki = []

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                przetwarzaj_plik,
                i
            ): i
            for i in numery_plikow
        }

        for future in as_completed(futures):

            i = futures[future]

            try:

                wynik = future.result()

                if wynik is not None:
                    wyniki.append(wynik)

            except Exception as e:

                print(
                    f"Błąd workera dla {i}.jpg: {e}"
                )

    # sortowanie po numerze zdjęcia
    wyniki.sort(
        key=lambda x: int(
            os.path.splitext(
                os.path.basename(
                    x["photo"]
                )
            )[0]
        )
    )

    return wyniki


# ============================================================
# 10. URUCHOMIENIE
# ============================================================

# ZMIEŃ ZAKRES NA SWÓJ
#
# np.:
# range(1, 101)
#
# oznacza:
# 1.jpg ... 100.jpg

numery = range(1, 27)

wyniki = przetworz_wszystkie(
    numery
)


# ============================================================
# 11. ZAPIS WYNIKÓW
# ============================================================

OUTPUT_PATH = (
    "/content/drive/MyDrive/wyniki.json"
)

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        wyniki,
        f,
        ensure_ascii=False,
        indent=2
    )


print()
print("=" * 60)
print(f"Gotowe. Przetworzono: {len(wyniki)} plików.")
print(f"Wyniki zapisane tutaj:")
print(OUTPUT_PATH)
print("=" * 60)


In [ ]:
ile_zdjec=28
if __name__ == "__main__":
    zakres = range(1, ile_zdjec + 1)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(executor.map(przetwarzaj_plik, zakres))

    final_data = [r for r in results if r is not None]

    with open('wyniki_egzaminu.json', 'w', encoding='utf-8') as f:
        json.dump(final_data, f, indent=4, ensure_ascii=False)

    print(f"\n Pobieranie gotowe! Przetworzono poprawnie {len(final_data)} plików.")

# Weryfikowanie

In [ ]:
import os
import json

from PIL import Image, ImageOps
from IPython.display import display
import ipywidgets as widgets


def weryfikuj_dane(
    plik_wejsciowy="/content/drive/MyDrive/wyniki.json",
    plik_wyjsciowy="/content/drive/MyDrive/wyniki_zweryfikowane.json"
):

    # =========================================================
    # WCZYTANIE
    # =========================================================

    if not os.path.exists(plik_wejsciowy):
        print(f"❌ Brak pliku: {plik_wejsciowy}")
        return

    with open(plik_wejsciowy, "r", encoding="utf-8") as f:
        wyniki = json.load(f)

    if not wyniki:
        print("❌ Plik wyników jest pusty.")
        return

    # =========================================================
    # TYLKO STRONY Z TYTUŁEM
    # =========================================================

    indeksy = [
        i
        for i, item in enumerate(wyniki)
        if item.get("tytul") is True
    ]

    if not indeksy:
        print("❌ Nie znaleziono żadnych prac z tytułem.")
        return

    # =========================================================
    # STAN
    # =========================================================

    aktualny = 0

    # =========================================================
    # WIDGETY
    # =========================================================

    obraz = widgets.Output(
        layout=widgets.Layout(
            width="100%"
        )
    )

    informacje = widgets.HTML(
        value="",
        layout=widgets.Layout(
            width="100%",
            padding="10px"
        )
    )

    pole_punktow = widgets.Text(
        value="",
        placeholder="Wpisz punkty lub zostaw puste",
        description="Punkty:",
        layout=widgets.Layout(
            width="350px"
        )
    )

    przycisk_zapisz = widgets.Button(
        description="✓ Zapisz / Dalej",
        button_style="success",
        layout=widgets.Layout(
            width="180px"
        )
    )

    przycisk_cofnij = widgets.Button(
        description="← Cofnij",
        button_style="warning",
        layout=widgets.Layout(
            width="140px"
        )
    )

    przycisk_zakoncz = widgets.Button(
        description="Zakończ i zapisz",
        button_style="danger",
        layout=widgets.Layout(
            width="180px"
        )
    )

    komunikat = widgets.HTML(
        value="",
        layout=widgets.Layout(
            width="100%",
            padding="10px"
        )
    )

    # =========================================================
    # POKAZANIE AKTUALNEGO ZDJĘCIA
    # =========================================================

    def pokaz():

        nonlocal aktualny

        indeks_json = indeksy[aktualny]
        item = wyniki[indeks_json]

        photo_path = item.get("photo")

        imie = item.get(
            "imie_i_nazwisko",
            "brak"
        )

        indeks = item.get(
            "nr_indeksu",
            "brak"
        )

        punkty = item.get("punkty")

        if punkty is None:
            punkty_text = "brak odczytu"
        else:
            punkty_text = str(punkty)

        # -----------------------------------------------------
        # INFORMACJE
        # -----------------------------------------------------

        informacje.value = f"""
        <div style="
            font-size:18px;
            line-height:1.6;
            border:1px solid #ccc;
            border-radius:8px;
            padding:10px;
        ">
            <b>Zdjęcie:</b>
            {aktualny + 1} / {len(indeksy)}
            &nbsp;&nbsp;|&nbsp;&nbsp;

            <b>Imię i nazwisko:</b>
            {imie}
            &nbsp;&nbsp;|&nbsp;&nbsp;

            <b>Nr indeksu:</b>
            {indeks}
            &nbsp;&nbsp;|&nbsp;&nbsp;

            <b>Punkty:</b>
            <span style="color:blue;">
                {punkty_text}
            </span>
        </div>
        """

        pole_punktow.value = ""
        komunikat.value = ""

        # -----------------------------------------------------
        # ZDJĘCIE
        # -----------------------------------------------------

        with obraz:

            obraz.clear_output(wait=True)

            if (
                photo_path
                and os.path.exists(photo_path)
            ):

                img = Image.open(photo_path)

                img = ImageOps.exif_transpose(img)

                img = img.convert("RGB")

                # Maksymalny rozmiar
                img.thumbnail((1200, 1200))

                szerokosc, wysokosc = img.size

                # Tylko górne 35% zdjęcia
                img = img.crop(
                    (
                        0,
                        0,
                        szerokosc,
                        int(wysokosc * 0.35)
                    )
                )

                # UWAGA:
                # NIE RYSUJEMY NIC NA ZDJĘCIU
                display(img)

            else:

                print(
                    f"❌ Nie znaleziono zdjęcia:\n"
                    f"{photo_path}"
                )

    # =========================================================
    # ZAPISZ / DALEJ
    # =========================================================

    def zapisz_dalej(b):

        nonlocal aktualny

        item = wyniki[indeksy[aktualny]]

        tekst = pole_punktow.value.strip()

        # -----------------------------------------------------
        # JEŻELI WPISANO NOWĄ WARTOŚĆ
        # -----------------------------------------------------

        if tekst:

            try:

                wartosc = float(
                    tekst.replace(",", ".")
                )

                if not 0 <= wartosc <= 30:

                    komunikat.value = """
                    <span style="color:red;">
                        ❌ Punkty muszą być od 0 do 30.
                    </span>
                    """

                    return

                if wartosc.is_integer():
                    item["punkty"] = int(wartosc)
                else:
                    item["punkty"] = wartosc

            except ValueError:

                komunikat.value = """
                <span style="color:red;">
                    ❌ Wpisz prawidłową liczbę.
                </span>
                """

                return

        # -----------------------------------------------------
        # NASTĘPNE
        # -----------------------------------------------------

        if aktualny < len(indeksy) - 1:

            aktualny += 1
            pokaz()

        else:

            zakoncz()

    # =========================================================
    # COFNIJ
    # =========================================================

    def cofnij(b):

        nonlocal aktualny

        if aktualny > 0:

            aktualny -= 1
            pokaz()

        else:

            komunikat.value = """
            <span style="color:#cc6600;">
                ⚠️ To jest pierwsza praca.
            </span>
            """

    # =========================================================
    # ZAKOŃCZ
    # =========================================================

    def zakoncz(b=None):

        with open(
            plik_wyjsciowy,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                wyniki,
                f,
                indent=4,
                ensure_ascii=False
            )

        komunikat.value = f"""
        <div style="
            background:#d4edda;
            color:#155724;
            padding:15px;
            border-radius:8px;
            font-size:18px;
        ">
            ✅ <b>Weryfikacja zakończona!</b><br><br>
            Zapisano:<br>
            {plik_wyjsciowy}
        </div>
        """

        pole_punktow.disabled = True
        przycisk_zapisz.disabled = True
        przycisk_cofnij.disabled = True
        przycisk_zakoncz.disabled = True

    # =========================================================
    # PRZYCISKI
    # =========================================================

    przycisk_zapisz.on_click(zapisz_dalej)
    przycisk_cofnij.on_click(cofnij)
    przycisk_zakoncz.on_click(zakoncz)

    # =========================================================
    # INTERFEJS
    # =========================================================

    przyciski = widgets.HBox(
        [
            przycisk_cofnij,
            przycisk_zapisz,
            przycisk_zakoncz
        ],
        layout=widgets.Layout(
            width="100%",
            justify_content="center"
        )
    )

    panel = widgets.VBox(
        [
            obraz,
            informacje,

            widgets.HTML(
                "<hr><h3>WERYFIKACJA PUNKTÓW</h3>"
            ),

            pole_punktow,

            widgets.HTML(
                "Wpisz poprawną liczbę punktów "
                "albo zostaw pole puste."
            ),

            przyciski,
            komunikat
        ],
        layout=widgets.Layout(
            width="100%"
        )
    )

    # =========================================================
    # START
    # =========================================================

    display(panel)

    pokaz()


# =============================================================
# URUCHOMIENIE
# =============================================================

weryfikuj_dane()


In [ ]:
weryfikuj_dane()


# Dopasowywanie

In [ ]:
import json
from difflib import SequenceMatcher

def levenshtein_dist(s1, s2):
    """Oblicza liczbę edycji (literówek) między dwoma ciągami."""
    s1, s2 = str(s1), str(s2)
    if len(s1) < len(s2):
        return levenshtein_dist(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def get_name_sim(n1, n2):
    """Zwraca podobieństwo nazwisk (0.0 - 1.0)."""
    return SequenceMatcher(None, str(n1).lower(), str(n2).lower()).ratio()

def napraw_wyniki(wyniki, lista):
    for w_row in wyniki:
        w_idx = str(w_row.get('nr_indeksu', ""))
        w_name = str(w_row.get('imie_i_nazwisko', ""))

        finalny_match = None

        # --- KROK 1: Indeks zgadza się idealnie ---
        for l_row in lista:
            if w_idx == str(l_row['nr_indeksu']) and w_idx != "":
                finalny_match = l_row
                break

        # --- KROK 2: Indeks ma 1-3 błędy -> Szukamy najlepszego nazwiska w tej grupie ---
        if not finalny_match:
            potencjalni_po_indeksie = []
            for l_row in lista:
                dist = levenshtein_dist(w_idx, str(l_row['nr_indeksu']))
                if dist <= 3:
                    score = get_name_sim(w_name, l_row['imie_i_nazwisko'])
                    potencjalni_po_indeksie.append((score, l_row))

            if potencjalni_po_indeksie:
                # Wybieramy tego, kto przy podobnym indeksie ma najsensowniejsze nazwisko
                potencjalni_po_indeksie.sort(key=lambda x: x[0], reverse=True)
                finalny_match = potencjalni_po_indeksie[0][1]

        # --- KROK 3: Indeks to totalna porażka -> Szukamy tylko po nazwisku ---
        if not finalny_match:
            wszyscy_po_nazwisku = []
            for l_row in lista:
                score = get_name_sim(w_name, l_row['imie_i_nazwisko'])
                if score > 0.7: # Próg bezpieczeństwa, żeby nie przypisać "Abc" do "Xyz"
                    wszyscy_po_nazwisku.append((score, l_row))

            if wszyscy_po_nazwisku:
                wszyscy_po_nazwisku.sort(key=lambda x: x[0], reverse=True)
                finalny_match = wszyscy_po_nazwisku[0][1]

        # --- NADPISYWANIE DANYCH ---
        if finalny_match:
            w_row['imie_i_nazwisko'] = finalny_match['imie_i_nazwisko']
            w_row['nr_indeksu'] = finalny_match['nr_indeksu']
        else:
            w_row['imie_i_nazwisko'] = "nie_znaleziono"
            w_row['nr_indeksu'] = "nie_znaleziono"

    return wyniki


# Zrób tak:
with open("wyniki_egzaminu.json", "r", encoding="utf-8") as f:
    wyniki_json = json.load(f)

with open("lista_studentow.json", "r", encoding="utf-8") as f:
    lista_json = json.load(f)

naprawione = napraw_wyniki(wyniki_json, lista_json)
print(json.dumps(naprawione, indent=4, ensure_ascii=False))